# 3.2 Scaling: what more weights, data and compute buy

At the end of this notebook you can:

- **measure** how the score of a model moves with its number of weights, with the amount of
  training data and with the compute spent on training, over a few seeds, and say which
  differences are larger than the spread between them;
- **look beyond the top score**: name a smaller model that is nearly as good and faster, and
  state its price;
- **read a scaling law**: a straight line on log-log axes whose slope says what ten times more
  of a resource buys, and only inside the range that was measured; say that it was measured in
  language, images and speech, with a slope that differs per domain;
- **read from a measured ratio** how much more data a bigger model needs.

Every study you run on the heartbeats spends a budget on one of three things: a bigger model,
more data, longer training. A **scaling curve**, the score against one resource with the
others held fixed, tells you which of the three still pays, before you spend the budget.

The measurements run on DTD (photographs of textures, 47 classes), as cached `dinov2-small`
vectors, the dataset of `1_representation_and_head/04_search`. The encoder is frozen: only the
head trains, so a trained point costs about a second, and you can draw the curves yourself.

**Three splits.** Every choice (an epoch count, the head to prefer) is made on a **valid**
split cut from the cache's train split. Every curve and every number quoted about it is scored
on the cache's **test** split, which no choice has looked at.

**How this notebook runs.** The DTD experiments train live. A second dataset's data axis is
trained by `eurosat_data.py`, which stores `results/eurosat_data.json` for this notebook to
read; `RERUN = True`, in the cell that reads it, retrains it in under half a minute.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from notebooktester import param
from scaling import choose_epochs, data_axis, load, mean_std, repeat
from torch import nn
from vectormesh.components import NeuralNet

SEEDS = param((0, 1, 2), test=(0, 1))

dtd = load("dtd", "dinov2-small")
DIM = dtd["train"][0].shape[1]
K = int(dtd["train"][1].max()) + 1

print(f"DTD, dinov2-small: {DIM}-dim vectors, {K} classes")
for name, (X, y) in dtd.items():
    per_class = torch.bincount(y, minlength=K)
    lo, hi = int(per_class.min()), int(per_class.max())
    count = f"{lo}" if lo == hi else f"{lo} to {hi}"
    print(f"  {name:5s} {tuple(X.shape)}   {count} rows per class")

The tools live in `scaling.py`, next to this notebook:

- `train` trains a head with Adam, batch 128, for a number of **epochs** (passes over the
  train rows) or a number of **optimizer steps** (one step is one batch: a forward pass, a
  backward pass, one update of the weights). It records the train and valid loss and the valid
  and test accuracy after every epoch.
- `repeat` runs `train` once per seed in `SEEDS`, three seeds.
- `choose_epochs` is the rule of 3.2.1 that picks an epoch count on valid.
- `mean_std` prints a mean ± std.

The ± is the std over the seeds and shows how far one run can move; runs from other random
weights converge, so a large spread first asks for longer training. The seeds vary the initial
weights and the order of the batches, not the split or which rows a subset holds.

## 3.2.1 How long is long enough, and what does compute buy?

**Question.** Every point in this notebook is a trained head. A head trained too briefly
measures its training budget, not the resource the curve is about. So: after how many epochs
has the head finished learning? And past that point, does more compute still pay?

**Setup.** The baseline head, `NeuralNet(384, 47)` (one hidden layer of 384 units), trained
for `MAX_EPOCHS = 80` epochs with three seeds. The rule: `EPOCHS` is the epoch with the
highest valid accuracy, averaged over the seeds. That is the epoch early stopping on valid
accuracy keeps (`1_representation_and_head/03_training`), applied to the mean of three runs.

**Expect.** Write down: at what epoch does valid accuracy stop rising, and does valid loss
stop falling at the same epoch?

In [ ]:
MAX_EPOCHS = param(80, test=2)

base = repeat(lambda: NeuralNet(DIM, K), dtd, SEEDS, epochs=MAX_EPOCHS)
EPOCHS = choose_epochs(base["valid"])

print(f"{'epoch':>5s} {'train loss':>18s} {'valid loss':>12s} {'valid accuracy':>15s}")
for e in [1, 2, 5, 10, 20, 40, 80]:
    if e <= MAX_EPOCHS:
        print(
            f"{e:5d} {mean_std(base['train_loss'][:, e - 1]):>18s} "
            f"{mean_std(base['valid_loss'][:, e - 1]):>12s} {mean_std(base['valid'][:, e - 1]):>15s}"
        )
low = int(base["valid_loss"].mean(0).argmin()) + 1
print(
    f"EPOCHS = {EPOCHS}: the highest mean valid accuracy; the lowest mean valid loss is at epoch {low}"
)

ep = np.arange(1, MAX_EPOCHS + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
a1.plot(ep, base["train_loss"].mean(0), label="train loss")
a1.plot(ep, base["valid_loss"].mean(0), label="valid loss")
a1.set_yscale("log")
a1.set_ylabel("cross-entropy loss (log)")
a1.set_title("Train loss keeps falling; valid loss turns up early")
for s, run in zip(SEEDS, base["valid"]):
    a2.plot(
        ep,
        run,
        color="tab:orange",
        lw=0.8,
        alpha=0.5,
        label="one seed" if s == SEEDS[0] else None,
    )
a2.plot(ep, base["valid"].mean(0), color="tab:orange", lw=2, label="mean of the seeds")
a2.set_ylabel("valid accuracy")
a2.set_title("Valid accuracy flattens and stays flat")
for a in (a1, a2):
    a.axvline(EPOCHS, color="grey", ls="--", label=f"EPOCHS = {EPOCHS}")
    a.set_xscale("log")
    a.set_xlabel("epoch (log)")
    a.grid(alpha=0.3)
    a.legend(frameon=False)
plt.tight_layout()
plt.show()

**Reading.** Valid accuracy rises to about epoch 10 (0.801 ± 0.005), reaches its highest mean
at epoch 20 (0.804 ± 0.004) and then stays near 0.80 up to epoch 80. Epochs 10 and 20 differ
by less than the spread between seeds: 20 is the rule's pick, not a sharp optimum.

Valid **loss** tells another story: lowest at epoch 5 (0.69 ± 0.01), then rising to 1.01 ±
0.01 at epoch 80 while train loss falls below 0.0005, the overfitting of lesson 1. The head
memorises the train rows and grows more confident on the valid rows it gets wrong, but how
many rows it gets right hardly changes. This notebook reports accuracy, so the epoch is chosen
on accuracy: **`EPOCHS = 20`**.

**The same runs as a compute curve.** Compute here is the number of optimizer steps of one
head: 24 per epoch on the 3008 train rows. The runs above also recorded their test accuracy
after every epoch, which played no part in choosing `EPOCHS`; read against the steps, that is
the compute axis.

**Expect.** Write down: does the test accuracy rise between `EPOCHS` and `MAX_EPOCHS`, four
times the steps, by more than the spread?

In [ ]:
steps = base["steps"][0]
print(f"{'epoch':>5s} {'steps':>6s} {'test accuracy':>15s}")
for e in [1, 2, 5, 10, 20, 40, 80]:
    if e <= MAX_EPOCHS:
        print(f"{e:5d} {int(steps[e - 1]):6d} {mean_std(base['test'][:, e - 1]):>15s}")
a, b = base["test"][:, EPOCHS - 1], base["test"][:, -1]
print(
    f"test accuracy, epoch {EPOCHS} -> {MAX_EPOCHS}: {mean_std(a)} -> {mean_std(b)}, "
    f"{100 * (b.mean() - a.mean()):+.1f} points"
)

plt.figure(figsize=(7, 3.8))
for s, run in zip(SEEDS, base["test"]):
    plt.plot(
        steps,
        run,
        color="tab:green",
        lw=0.8,
        alpha=0.5,
        label="one seed" if s == SEEDS[0] else None,
    )
plt.plot(
    steps, base["test"].mean(0), color="tab:green", lw=2, label="mean of the seeds"
)
plt.axvline(steps[EPOCHS - 1], color="grey", ls="--", label=f"epoch {EPOCHS}")
plt.xscale("log")
plt.xlabel("optimizer steps (log)")
plt.ylabel("test accuracy")
plt.title("Compute pays at first, then the curve is flat")
plt.grid(alpha=0.3)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

**Reading.** Compute pays at first and then stops. One epoch (24 steps) gives 0.74 ± 0.01 on
test, epoch 5 (120 steps) 0.786 ± 0.005, epoch 20 (480 steps) 0.801 ± 0.005. From epoch 20 to
80, four times the steps, the head gains 0.1 points, smaller than the spread between seeds
(0.4 to 0.5 points). The test curve agrees with the valid curve, which chose epoch 20 without
seeing it.

## 3.2.2 Parameters

**Question.** Does a head with more weights score higher? The heads grow in **width**: the
linear map, then one hidden layer of 64 to 1024 units (`1 x 64` is one hidden layer of 64).
The baseline of 3.2.1, `NeuralNet(384, 47)`, is `1 x 384` on this axis.

**Setup.** All 3008 train rows, three seeds per head, each head trained for 40 epochs. A wider
head could need longer than the baseline, so each head gets its own epoch, chosen on valid by
the rule of 3.2.1, and its test accuracy is read at that epoch.

**Expect.** Write down: is the 1024-wide head better than the linear one by more than the
spread?

In [ ]:
HEAD_EPOCHS = param(40, test=2)


def head(width: int | None = None):
    # NeuralNet with one hidden layer of this width; no width is one Linear layer.
    if width is None:
        return lambda: nn.Linear(DIM, K)
    return lambda: NeuralNet([DIM, width], K)


HEADS = {"linear": head(), **{f"1 x {w}": head(w) for w in (64, 128, 256, 512, 1024)}}

heads = {}
for name, make in HEADS.items():
    heads[name] = repeat(make, dtd, SEEDS, epochs=HEAD_EPOCHS)
    heads[name]["epoch"] = choose_epochs(heads[name]["valid"])


def at_own_epoch(name: str, key: str = "test") -> np.ndarray:
    # One head's accuracy per seed, at its own epoch chosen on valid.
    return heads[name][key][:, heads[name]["epoch"] - 1]


print(f"{'head':9s} {'params':>9s} {'epoch':>5s}   test accuracy")
for name in HEADS:
    print(
        f"{name:9s} {heads[name]['params']:9,d} {heads[name]['epoch']:5d}   {mean_std(at_own_epoch(name))}"
    )
d = at_own_epoch("1 x 1024").mean() - at_own_epoch("linear").mean()
print(f"test accuracy, linear -> 1 x 1024: {100 * d:+.1f} points")

params = [heads[n]["params"] for n in HEADS]
runs = np.array([at_own_epoch(n) for n in HEADS])
plt.figure(figsize=(7, 3.8))
plt.errorbar(
    params,
    runs.mean(1),
    yerr=runs.std(1),
    fmt="o-",
    capsize=3,
    label="mean ± std over the seeds",
)
for p, r in zip(params, runs):
    plt.plot([p] * len(r), r, ".", color="grey", alpha=0.6)
plt.plot([], [], ".", color="grey", label="one seed")
plt.xscale("log")
plt.xticks(params, [f"{n}\n{p:,}" for n, p in zip(HEADS, params)], fontsize=7)
plt.minorticks_off()
plt.xlabel("head and parameters (log)")
plt.ylabel("test accuracy at each head's own epoch")
plt.title("Width buys a little")
plt.grid(alpha=0.3)
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

**Reading.** Width buys a little. From the linear map (0.776 ± 0.003, 18,095 parameters) to
`1 x 1024` (0.802 ± 0.002, 442,415) is +2.5 points, about ten times the spread between seeds.
Most of it has come by `1 x 512` (0.800 ± 0.003); the last doubling adds less than the spread.

Keep these two and a half points in mind for the data axis of 3.2.3.

**Is the comparison fair?** A head scored while it was still learning would measure its
budget. The curves of the smallest head, the widest, and `1 x 256` between them show where each
head's epoch falls.

In [ ]:
hep = np.arange(1, HEAD_EPOCHS + 1)
plt.figure(figsize=(7, 3.8))
for i, n in enumerate(["linear", "1 x 256", "1 x 1024"]):
    r, e = heads[n], heads[n]["epoch"]
    plt.plot(hep, r["valid"].mean(0), color=f"C{i}", label=f"{n}, epoch {e}")
    plt.plot(e, r["valid"].mean(0)[e - 1], "o", color=f"C{i}")
plt.ylabel("valid accuracy, mean of the seeds")
plt.title("Valid accuracy has flattened by each head's epoch (the dot)")
plt.xscale("log")
plt.xlabel("epoch (log)")
plt.grid(alpha=0.3)
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

Every valid-accuracy curve has flattened by its dot and stays near that level to epoch 40: no
head was scored while it was still learning. The three keep their order the whole way,
the widest on top.

**Beyond the top score.** A head close to the best, with fewer parameters, does nearly the
same job for less. Choosing it is a choice, so it is made on **valid**. The next cell prints
the valid accuracy of every head at its own epoch.

**Expect.** Write down which heads lie close to the best on valid, and which you would choose.

In [ ]:
BEST = max(HEADS, key=lambda n: at_own_epoch(n, "valid").mean())
print(f"{'head':9s} {'params':>9s}   valid accuracy")
for n in HEADS:
    print(f"{n:9s} {heads[n]['params']:9,d}   {mean_std(at_own_epoch(n, 'valid'))}")
print(f"best on valid: {BEST}")

**Reading.** On valid, `1 x 1024` is best (0.808 ± 0.003). `1 x 256` (0.799 ± 0.007) and
`1 x 512` (0.801 ± 0.004) lie close to it, below it by about the spread between seeds; another
seed could change their order. The smallest of those, `1 x 256`, has a quarter of the
parameters (110,639 against 442,415): it is the choice. The next cell reads both on test,
once, and times one forward pass over the 1880 test rows, repeated (median and middle half).

**Expect.** Write down whether a quarter of the parameters makes `1 x 256` four times as fast.

In [ ]:
REPEATS = param(50, test=3)


def latency_ms(make, X: torch.Tensor) -> tuple[float, float, float]:
    # Median, 25th and 75th percentile of the time of one forward pass over X, in ms.
    torch.manual_seed(0)
    model = make().eval()
    times = []
    with torch.no_grad():
        model(X)  # the first call allocates; it is not timed
        for _ in range(REPEATS):
            start = time.perf_counter()
            model(X)
            times.append(1000 * (time.perf_counter() - start))
    return tuple(float(v) for v in np.percentile(times, [50, 25, 75]))


PICK = "1 x 256"  # the smallest of the heads close to the best on valid

d = at_own_epoch(PICK).mean() - at_own_epoch(BEST).mean()
print(f"test accuracy, read once, {BEST} -> {PICK}: {100 * d:+.1f} points")
print(
    f"{'head':9s} {'params':>9s} {'test accuracy':>15s} {'ms per pass (25-75%)':>24s}"
)
ms = {}
for n in dict.fromkeys([BEST, PICK]):
    ms[n], lo, hi = latency_ms(HEADS[n], dtd["test"][0])
    print(
        f"{n:9s} {heads[n]['params']:9,d} {mean_std(at_own_epoch(n)):>15s} {ms[n]:9.2f} ({lo:.2f} to {hi:.2f})"
    )
print(f"{BEST} takes {ms[BEST] / ms[PICK]:.1f} times as long per pass as {PICK}")

**Reading.** Read on test, once, `1 x 256` costs about half a point (0.796 ± 0.003 against
0.802 ± 0.002). That is the price of the choice. It is reported, not undone: choosing again on
test would make the test number a choice too.

On the machine that ran this notebook, one pass over the 1880 test rows took `1 x 1024` between
three and four times as long as `1 x 256`; the middle halves of the repeats do not overlap,
and the milliseconds change from run to run. So, with its price stated:
**`1 x 256` answers three to four times as fast for about half a point of test accuracy**, on this
machine and a batch of 1880 rows. Energy was not measured: fewer operations suggest less, but
a claim about energy needs an energy measurement.

## 3.2.3 Data

**Question.** Does the score move with the number of labelled rows?

**Setup.** The baseline head on 5%, 10%, 25%, 50% and 100% of the train rows, three seeds
each, scored on test: from 150 to 3008 rows, a factor of 20, close to the 24 of the parameter
axis. A subset is a prefix of the train rows, which come in a **balanced order**: every prefix
holds each class about equally often (the first 150 rows hold 3 or 4 of every class), and a
smaller prefix is part of every larger one. Stored class by class, a prefix would hold only
the first classes.

At a fixed number of epochs, 5% of the rows is 2 batches per epoch and 100% is 24, so the
small subset also gets a twelfth of the optimizer steps: data and compute shrink together. So
the axis is measured twice:

- **equal epochs**: every subset gets `EPOCHS` passes, as when you keep your recipe and
  collect fewer rows.
- **equal steps**: every subset gets the optimizer steps of the full set at `EPOCHS` epochs,
  so a small subset is passed over many more times. Compute is fixed; only the data changes.

At 100% both arms are the baseline runs of 3.2.1, read at `EPOCHS`.

**Expect.** Write down: how far apart are 5% and 100%? And does giving the small subsets the
full number of steps close the gap?

In [ ]:
FRACTIONS = param([0.05, 0.1, 0.25, 0.5, 1.0], test=[0.25, 1.0])
ARMS = {
    "equal_epochs": "equal epochs: fewer rows, fewer steps",
    "equal_steps": "equal steps: fewer rows, more passes over them",
}


def show_data_axis(rows: list[dict], k: int, label: str) -> None:
    print(
        f"{label}: {'rows':>5s} {'per class':>9s} | {'steps':>5s} {'equal epochs':>16s} | {'steps':>5s} {'equal steps':>16s}"
    )
    for r in rows:
        print(
            f"{'':{len(label) + 1}s} {r['rows']:5d} {r['rows'] / k:9.1f} | {r['steps_equal_epochs']:5d} "
            f"{mean_std(r['test_equal_epochs']):>16s} | {r['steps_equal_steps']:5d} {mean_std(r['test_equal_steps']):>16s}"
        )


def plot_data_axis(rows: list[dict], title: str) -> None:
    n = np.array([r["rows"] for r in rows])
    plt.figure(figsize=(7, 3.8))
    for (arm, label), fmt, shift in zip(ARMS.items(), ["o-", "s--"], [0.97, 1.03]):
        runs = np.array([r[f"test_{arm}"] for r in rows])
        bars = plt.errorbar(
            n * shift,
            runs.mean(1),
            yerr=runs.std(1),
            fmt=fmt,
            capsize=3,
            label=f"{label}, mean ± std",
        )
        plt.plot(
            np.repeat(n * shift, runs.shape[1]),
            runs.ravel(),
            ".",
            color=bars[0].get_color(),
            alpha=0.4,
        )
    plt.xscale("log")
    plt.xticks(n, [str(v) for v in n])
    plt.minorticks_off()
    plt.xlabel("training rows (log)")
    plt.ylabel("test accuracy")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()


def compare_data_axis(rows: list[dict]) -> None:
    # The two comparisons of a reading: the arms at the smallest subset, the last doubling.
    small, last, full = rows[0], rows[-2], rows[-1]
    for text, a, b in [
        (
            f"at {small['rows']} rows, equal epochs -> equal steps",
            small["test_equal_epochs"],
            small["test_equal_steps"],
        ),
        (
            f"the last doubling, {last['rows']} -> {full['rows']} rows at equal steps",
            last["test_equal_steps"],
            full["test_equal_steps"],
        ),
    ]:
        print(
            f"{text}: {mean_std(a)} -> {mean_std(b)}, {100 * (np.mean(b) - np.mean(a)):+.1f} points"
        )


dtd_data = data_axis(dtd, base, EPOCHS, FRACTIONS, SEEDS)
show_data_axis(dtd_data, K, "DTD")
compare_data_axis(dtd_data)
plot_data_axis(dtd_data, "DTD: the score moves with the data, not with the steps")

**Reading.** At equal epochs the curve runs from 0.58 ± 0.01 at 150 rows (3.2 per class, 40
steps) to 0.801 ± 0.005 at 3008 rows (480 steps): 22 points, nine times the whole parameter
axis.

Is that the data, or the twelve times as many steps? The equal-steps arm answers it. Giving
the 150 rows all 480 steps lifts them by 1.3 points, about the spread between seeds (1 point):
another seed could turn this around. From 300 rows on, the two arms lie on top of each other.
At equal compute the curve still spans 20.5 points, so almost all of the movement is data.

The last doubling, from 1504 to 3008 rows at equal steps, still adds 3.4 points, far more than
the spread (0.4 to 0.5 points).
**On DTD, more labelled rows have not stopped paying at the full train split.** At 3 rows per
class the head already reaches 0.58, against a chance level of 1/47 ≈ 0.02: that start is the
frozen encoder's doing.

### The three axes side by side

DTD's three curves, in points of test accuracy; the **last step** says whether an axis still
pays at its end.

| axis | measured over | last step |
|---|---|---|
| parameters | linear to `1 x 1024` | `1 x 512` to `1 x 1024`: less than the spread |
| data | 150 to 3008 rows, equal steps | 1504 to 3008 rows: +3.4, far more than the spread |
| compute | 1 to 80 epochs, 24 to 1920 steps | epoch 20 to 80: +0.1, less than the spread |

Of the three axes, only data still pays at its end. The compute and width curves flatten
because the model, the data or the frozen features are the limit; Kaplan's laws, next, hold
only where no other resource is.

## 3.2.4 Reading a scaling law: Kaplan's three laws

**Question.** The scaling-law papers draw the same three axes for models trained from scratch,
over far wider ranges, and fit a law to them. How do you read such a law, and what does it not
promise? The worked example is Kaplan et al. (2020), *Scaling Laws for Neural Language Models*
(the PDF in `docs/papers/`).

**A power law is a straight line on log-log axes.** A law of the form
$L(X) = (X_c / X)^{\alpha}$ gives, after taking the log of both sides,
$\log L = \alpha \log X_c - \alpha \log X$: a straight line in $\log X$ with slope $-\alpha$.
So ×10 on the x axis is ×$10^{-\alpha}$ on the y axis. The slope is what you read.

**What was measured.** Transformer language models trained on WebText2, a large collection of
web text. The score is the **test loss**: the cross-entropy per token on held-out text, in
nats (the natural logarithm, as `nn.CrossEntropyLoss` uses); lower is better. Three resources,
each with its unit:

- **parameters $N$**, non-embedding: the weights without the embeddings, the lookup tables
  that turn each token into a vector;
- **dataset size $D$**, in tokens (a token is a word or a piece of one);
- **compute $C_{min}$**, in PF-days: $10^{15}$ floating point operations per second for a day,
  $8.64 \cdot 10^{19}$ operations. Kaplan does not train one model longer: for each compute
  budget he takes the model size that reaches the lowest loss with it, which differs from the
  compute axis of 3.2.1 (steps of one fixed head, which flattens).

Each law holds only when the other two resources are not the bottleneck:

| law | holds for | $\alpha$ | $X_c$ | equation |
|---|---|---|---|---|
| $L(N)$ | models trained to convergence on enough data | 0.076 | $8.8 \cdot 10^{13}$ parameters | 1.1 |
| $L(D)$ | large models on a limited dataset, early stopping | 0.095 | $5.4 \cdot 10^{13}$ tokens | 1.2 |
| $L(C_{min})$ | enough data, a model of the best size | 0.050 | $3.1 \cdot 10^{8}$ PF-days | 1.3 |

The constants $X_c$ depend on how the text is cut into tokens and have no meaning of their
own. Each line below is the fitted law, not the runs, drawn only over the range of the runs
behind it: compute from about $8 \cdot 10^{-9}$ to $0.25$ PF-days, dataset size from
$2.1 \cdot 10^{7}$ to $1.4 \cdot 10^{9}$ tokens, parameters from about $6 \cdot 10^{3}$ to
$1.5 \cdot 10^{9}$.

In [ ]:
# Kaplan et al. (2020): alpha and X_c of eq. 1.1 to 1.3; lo and hi are the measured range.
KAPLAN = {
    "C": {
        "alpha": 0.050,
        "Xc": 3.1e8,
        "lo": 8e-9,
        "hi": 0.25,
        "eq": "1.3",
        "xlabel": "compute $C_{min}$ (PF-days, log)",
    },
    "D": {
        "alpha": 0.095,
        "Xc": 5.4e13,
        "lo": 2.1e7,
        "hi": 1.4e9,
        "eq": "1.2",
        "xlabel": "dataset size $D$ (tokens, log)",
    },
    "N": {
        "alpha": 0.076,
        "Xc": 8.8e13,
        "lo": 6e3,
        "hi": 1.5e9,
        "eq": "1.1",
        "xlabel": "parameters $N$ (non-embedding, log)",
    },
}


def kaplan_loss(axis: str, x) -> np.ndarray:
    # Test loss in nats per token from the fitted power law L = (X_c / x)^alpha.
    k = KAPLAN[axis]
    return (k["Xc"] / np.asarray(x, dtype=float)) ** k["alpha"]


fig, axs = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for a, (axis, k) in zip(axs, KAPLAN.items()):
    x = np.geomspace(k["lo"], k["hi"], 200)
    a.plot(x, kaplan_loss(axis, x), color="tab:blue")
    a.plot(
        [k["lo"], k["hi"]],
        kaplan_loss(axis, [k["lo"], k["hi"]]),
        "o",
        color="tab:blue",
        label="ends of the measured range",
    )
    a.set_xscale("log")
    a.set_yscale("log")
    a.set_xlabel(k["xlabel"])
    a.set_title(f"$L({axis})$, eq. {k['eq']}: slope $-{k['alpha']}$", fontsize=10)
    a.grid(alpha=0.3, which="both")
    # every panel spans the same 8 decades, so the angles of the lines compare by eye
    mid = (np.log10(k["lo"]) + np.log10(k["hi"])) / 2
    a.set_xlim(10 ** (mid - 4), 10 ** (mid + 4))
ticks = [2.4, 3.0, 4.0, 5.0, 6.0, 7.0]
axs[0].set_yticks(ticks)
axs[0].set_yticklabels([f"{t:g}" for t in ticks])
axs[0].yaxis.set_minor_formatter(plt.NullFormatter())
axs[0].set_ylabel("test loss (nats per token, log)")
axs[0].legend(frameon=False, fontsize=8)
plt.suptitle(
    "Kaplan et al. (2020): every resource gives a straight line on log-log axes, drawn only where measured"
)
plt.tight_layout()
plt.show()

Three straight lines: each resource, grown by a factor, cuts the loss by a fixed factor over
the whole measured range. Every panel spans the same eight decades on its x axis, so the
angles of the lines compare by eye.

**Expect.** Read two values off the figure before you run the next cell: the test loss of a
model with $N = 10^{8}$ parameters (right panel), and of a large model trained on $D = 10^{9}$
tokens (middle panel). Then: which line is steepest, so buys the most per 10×?

In [ ]:
for axis, x in [("N", 1e8), ("D", 1e9)]:
    print(f"L({axis} = {x:.0e}) = {float(kaplan_loss(axis, x)):.2f} nats per token")
for axis, k in KAPLAN.items():
    print(f"10x more {axis}: loss x 10^-{k['alpha']} = {10 ** -k['alpha']:.2f}")

10× the data multiplies the loss by 0.80, 10× the parameters by 0.84, 10× the compute by 0.89:
the steeper the line, the more 10× buys.

**What the law does not promise.** Outside its measured range a line is a formula, not a
measurement. The next cell asks the parameter law about models far larger than any that was
trained.

In [ ]:
for n in [1.5e9, 1e12, 1e15, 1e20]:
    where = "inside" if n <= KAPLAN["N"]["hi"] else "outside"
    print(
        f"N = {n:.1e}: L = {float(kaplan_loss('N', n)):.2f} nats per token ({where} the measured range)"
    )

The formula keeps falling, below 1 nat per token and on towards 0. Kaplan et al. say the loss
must flatten out before that, because text cannot be predicted perfectly; where the bend
comes, the law does not say.

**When another resource is the bottleneck,** the line does not apply either. Kaplan's equation
1.5 fits $N$ and $D$ together, with constants of its own:
$L(N, D) = \left[(N_c/N)^{\alpha_N/\alpha_D} + D_c/D\right]^{\alpha_D}$. The next cell grows
the model on the full 22 billion tokens of WebText2 and on 86 million.

In [ ]:
# eq. 1.5 with the constants of its own fit (Kaplan et al. 2020, Table 2)
ND = {"aN": 0.076, "aD": 0.103, "Nc": 6.4e13, "Dc": 1.8e13}


def kaplan_loss_nd(n, d) -> float:
    # Early-stopped test loss of a model with n parameters on d tokens, eq. 1.5.
    return float(((ND["Nc"] / n) ** (ND["aN"] / ND["aD"]) + ND["Dc"] / d) ** ND["aD"])


print(f"{'N':>8s} {'L(N), eq. 1.1':>14s} {'D = 22e9':>9s} {'D = 86e6':>9s}")
for n in [1e6, 1e7, 1e8, 1e9]:
    print(
        f"{n:8.0e} {float(kaplan_loss('N', n)):14.2f} {kaplan_loss_nd(n, 22e9):9.2f} {kaplan_loss_nd(n, 86e6):9.2f}"
    )

On 22 billion tokens the loss falls with $N$ about as the parameter law says; it lies a little
below that column because equation 1.5 is a separate fit with its own constants. On 86 million
tokens the loss hardly moves past $10^{7}$ parameters: the data is the bottleneck.

## 3.2.5 A bigger model needs more data

**Question.** How much more data does a bigger model need to stay as close to its best as the
smaller one was?

On the same data each extra weight pays less, because the data becomes the limit. Trained on,
the test loss turns up: the overfitting of lesson 1. Early stopping takes the best point
before that, and for a bigger model that point lies less and less lower. The
**overfitting penalty** is the share of the loss lost to the limited data:
$\delta L = L(N, D) / L(N, \infty) - 1$, with $L(N, \infty)$ the loss of the same model on
unlimited data. Equation 1.5 at $D = \infty$ gives
$L(N, \infty) = \left[(N_c/N)^{\alpha_N/\alpha_D}\right]^{\alpha_D}$, so

$$\frac{L(N, D)}{L(N, \infty)} = \left[\frac{(N_c/N)^{\alpha_N/\alpha_D} + D_c/D}{(N_c/N)^{\alpha_N/\alpha_D}}\right]^{\alpha_D} = \left[1 + \frac{D_c}{N_c^{\alpha_N/\alpha_D}} \cdot \frac{N^{\alpha_N/\alpha_D}}{D}\right]^{\alpha_D}.$$

The penalty depends on $N$ and $D$ only through $N^{\alpha_N/\alpha_D} / D$. Keep that ratio
fixed and the penalty stays the same: the data has to grow as
$D \propto N^{0.076/0.103} = N^{0.74}$.

**Expect.** The model grows 8×. Write down: by what factor must the data grow to keep the
penalty equal, and what happens to the penalty if the data stays the same?

In [ ]:
exponent = ND["aN"] / ND["aD"]


def penalty(n, d) -> float:
    # Overfitting penalty of eq. 4.3: the loss on d tokens over the loss on unlimited data, minus 1.
    return float((1 + (n / ND["Nc"]) ** exponent * ND["Dc"] / d) ** ND["aD"] - 1)


grow = 8**exponent
print(f"alpha_N / alpha_D = {ND['aN']} / {ND['aD']} = {exponent:.2f}")
print(f"8x the parameters asks for 8^{exponent:.2f} = {grow:.1f}x the data\n")
n0, d0 = 1e8, 1e9
for label, n, d in [
    ("start", n0, d0),
    ("8x N, same D", 8 * n0, d0),
    (f"8x N, {grow:.1f}x D", 8 * n0, grow * d0),
]:
    print(
        f"{label:16s} N = {n:.1e}  D = {d:.1e}   loss {kaplan_loss_nd(n, d):.3f}   penalty {100 * penalty(n, d):.1f}% of the loss"
    )

**Reading.** With the same data, the 8× bigger model's loss is still lower (2.803 against
2.957 nats per token), but its penalty rises from 7.0% to 18.8% of the loss: a larger share of
its loss is lost to the limited data. With $8^{0.74} = 4.6$ times the data the penalty is back
at 7.0%, and the loss falls to 2.525. **8× the weights asks about 5× the data**, the number
Kaplan et al. give. The next figure draws both cases as the model grows.

In [ ]:
N = np.geomspace(1e7, 1.5e9, 100)
lines = {
    f"fixed data, D = {d0:.0e} tokens": d0 * np.ones_like(N),
    f"data grown as N^{exponent:.2f}": d0 * (N / n0) ** exponent,
}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
for (label, D), color in zip(lines.items(), ["tab:red", "tab:blue"]):
    a1.plot(N, [kaplan_loss_nd(n, d) for n, d in zip(N, D)], color=color, label=label)
    a2.plot(N, [100 * penalty(n, d) for n, d in zip(N, D)], color=color, label=label)
a1.plot(N, (ND["Nc"] / N) ** ND["aN"], color="grey", ls=":", label="unlimited data")
for a in (a1, a2):
    a.axvline(n0, color="grey", ls="--", lw=1)
    a.set_xscale("log")
    a.set_xlabel("parameters $N$ (log)")
    a.grid(alpha=0.3, which="both")
    a.legend(frameon=False, fontsize=8)
a1.set_ylabel("test loss, eq. 1.5 (nats per token)")
a1.set_title("On fixed data the loss still falls with N, ever slower")
a2.set_ylabel("overfitting penalty (% of the loss)")
a2.set_title("The penalty grows on fixed data and stays flat along D ~ N^0.74")
plt.tight_layout()
plt.show()

The dashed line is the start, $N = 10^{8}$ on $10^{9}$ tokens. In the left panel the penalty
is the gap between the fixed-data line (red) and the unlimited-data line (dotted), as a share
of the dotted one. On fixed data the loss still falls as the model grows, and the gap widens;
with the data grown as $N^{0.74}$ (blue) the penalty stays where it started.

**Two questions, two ratios.**

1. *How much data keeps the penalty equal?* Kaplan's answer, from equation 1.5: 8× the
   parameters asks for 4.6× the data.
2. *With a fixed compute budget, how to split it between weights and data?* A study measures
   the best model size as $N \propto C^{a}$ and the best data as $D \propto C^{b}$, so
   $D \propto N^{b/a}$. The next cell reads it for two studies.

In [ ]:
print("the best split of a compute budget (N ~ C^a, D ~ C^b, so D ~ N^(b/a)):")
# Hoffmann's three methods give a from 0.46 to 0.50 and b from 0.50 to 0.54.
for study, a, b in [("Kaplan", 0.73, 0.27), ("Hoffmann", 0.50, 0.50)]:
    print(
        f"  {study:8s} a = {a}, b = {b}: 8x the parameters goes with {8 ** (b / a):.1f}x the data"
    )

Kaplan et al. found $N \propto C^{0.73}$ and $D \propto C^{0.27}$: 8× the parameters goes with
2.2× the data. Hoffmann et al. (2022, arXiv:2203.15556) found $a$ and $b$ near 0.5: 8× the
parameters goes with about 8× the data, and three methods agree. The studies read their runs
differently. Kaplan read the loss of short runs from partway through long runs, before they
had finished learning; Hoffmann trained every run to its own end, and then more data paid off
more. At about the same compute, Hoffmann's Chinchilla (70 billion weights, 1.4 trillion
tokens) beat Gopher (280 billion weights, 300 billion tokens). A ratio belongs to the study
that measured it.

## 3.2.6 The same law in other domains, with other slopes

**Question.** Is a power law something of language models, or of one paper?

Two other studies measured it in other domains. Hestness et al. (2017, arXiv:1712.00409) grew
the training set in translation, image classification and speech recognition, and fitted the
validation error as $\varepsilon(m) \propto m^{\beta_g}$, with $m$ the number of training
examples. Henighan et al. (2020, arXiv:2010.14701) grew the model for generating images and
video, and fitted $L(N) = L_\infty + (N_0/N)^{\alpha_N}$: a power law on top of an
**irreducible loss** $L_\infty$ that no model gets below; the power law describes the part
above it, the reducible loss. The next cell lists one exponent per domain and draws the fitted
laws, not the runs; an exponent of $-0.3$ means 10× more multiplies the error by
$10^{-0.3} = 0.50$. The **top-1 error** is the share of images whose highest-scoring class is
wrong; the **character error rate** is the share of characters a transcript gets wrong.

In [ ]:
# domain, study, what grows, what falls, exponent
SLOPES = [
    ("language model", "Kaplan 2020", "data", "loss", -0.095),
    ("translation", "Hestness 2017", "data", "per-token error", -0.128),
    ("image classification", "Hestness 2017", "data", "top-1 error", -0.309),
    ("speech recognition", "Hestness 2017", "data", "character error rate", -0.299),
    ("image generation, 32x32", "Henighan 2020", "parameters", "reducible loss", -0.13),
    ("video generation", "Henighan 2020", "parameters", "reducible loss", -0.24),
]
print(
    f"{'domain':24s} {'study':14s} {'grows':10s} {'falls':20s} {'exponent':>8s} {'x per 10x':>9s}"
)
for domain, study, grows, falls, b in SLOPES:
    print(f"{domain:24s} {study:14s} {grows:10s} {falls:20s} {b:8.3f} {10**b:9.2f}")

OKABE_ITO = ["#E69F00", "#56B4E9", "#009E73", "#D55E00", "#0072B2", "#CC79A7"]
x = np.geomspace(1, 100, 50)
plt.figure(figsize=(7, 4))
for (domain, study, grows, _, b), color in zip(
    sorted(SLOPES, key=lambda s: s[4]), OKABE_ITO
):
    ls = "-" if grows == "data" else "--"
    # dashed lines on top, so a dashed line over a solid one with the same slope shows both
    z = 2 if grows == "data" else 3
    plt.plot(
        x,
        x**b,
        color=color,
        ls=ls,
        lw=2,
        zorder=z,
        label=f"{domain} ({grows}): slope {b}",
    )
plt.xscale("log")
plt.yscale("log")
plt.xticks([1, 10, 100], ["1", "10", "100"])
plt.yticks([0.25, 0.5, 0.75, 1.0], ["0.25", "0.50", "0.75", "1.00"])
plt.minorticks_off()
plt.xlabel("the resource, as a multiple of where it started (log)")
plt.ylabel("error or loss, relative to the start (log)")
plt.title("One form, six slopes: what 100x more buys differs per domain")
plt.grid(alpha=0.3, which="both")
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

**Reading.** In every domain the error or loss falls as a power law of the resource, but the
slope differs: 10× the data multiplies the loss of a language model by 0.80 and the top-1
error of image classification by 0.49. The metrics differ (an error rate, a cross-entropy, a
reducible loss), and so do the models and the training. The exponents say that the law has the
same form everywhere; they do not carry a slope from one domain to another.
**The slope of your problem is measured on your problem.**

## 3.2.7 Your own slope: DTD against EuroSAT

**Question.** The data axis of 3.2.3 is a curve of your own. On a second dataset the accuracy
can move more or fewer points over its data axis: does more movement in points mean a steeper
slope?

**Setup.** The second dataset is EuroSAT (satellite tiles, 10 land-use classes, the dataset of
`1_representation_and_head/03_training`), on its own `dinov2-small` cache. The script
`eurosat_data.py` runs the protocol of 3.2.1 and 3.2.3 on it; the next cell reads its results.

A scaling law describes a loss or an error that falls as the resource grows, down towards an
irreducible part such as the $L_\infty$ of 3.2.6. Here the **test error**, 1 − accuracy,
stands in for the loss.

**Expect.** Write down: EuroSAT starts with more rows per class and an easier task. Does its
accuracy move as many points over its data axis as DTD's?

In [ ]:
# True: train EuroSAT's data axis again with eurosat_data.py (about 20 seconds) before
# loading its results.
# Under notebooktester the script runs in a test mode of seconds and stores its results
# apart, in results/test.
RERUN = param(False, test=True)
RESULTS = Path("results", param("", test="test"))
if RERUN:
    subprocess.run([sys.executable, "eurosat_data.py"], check=True)
eurosat = json.loads((RESULTS / "eurosat_data.json").read_text())
eurosat_data = eurosat["data_axis"]

print(
    f"EuroSAT, dinov2-small: {eurosat['train_rows']} train rows, {eurosat['classes']} classes, "
    f"{len(eurosat['seeds'])} seeds; epochs chosen on valid: {eurosat['epochs']} of {eurosat['max_epochs']}"
)
show_data_axis(eurosat_data, eurosat["classes"], "EuroSAT")
compare_data_axis(eurosat_data)
for name, rows in [("DTD", dtd_data), ("EuroSAT", eurosat_data)]:
    means = [np.mean(r["test_equal_steps"]) for r in rows]
    print(
        f"{name}: test accuracy at equal steps moves {100 * (means[-1] - means[0]):.1f} points from {rows[0]['rows']} to {rows[-1]['rows']} rows"
    )

**Reading.** EuroSAT starts high and moves little. At equal steps its accuracy runs from 0.890
± 0.002 at 240 rows (24 per class) to 0.949 ± 0.003 at 4800: 5.9 points, against 20.5 on DTD.
At 240 rows the extra steps lift it by 0.9 points, more than the spread between seeds (0.2 to
0.4 points), so at the smallest subset part of the movement is compute; from 2400 rows on the
two arms lie on top of each other. The last doubling adds 0.4 points at equal steps, about the
spread, against 3.4 on DTD. In points of accuracy, more data pays far less on EuroSAT. The
next cell asks whether that is another slope or less room to fall.

**On log-log axes.** The next cell draws the test error of the equal-steps arm (only the data
changes) against the number of training rows, both on log axes, and fits a straight line per
dataset and per seed: $\log_{10}(\text{error}) = c + \beta \log_{10}(\text{rows})$. It prints
the slope $\beta$ as mean ± std over the seeds, and what 10× more rows multiplies the error
by.

**Expect.** Write down: will the two slopes match, or is DTD's much steeper?

In [ ]:
def own_slopes(rows: list[dict]) -> tuple:
    # Rows, test error per row count and seed, and the fitted slope and intercept per seed.
    n = np.array([r["rows"] for r in rows], dtype=float)
    err = 1 - np.array([r["test_equal_steps"] for r in rows])
    fit = np.array([np.polyfit(np.log10(n), np.log10(e), 1) for e in err.T])
    return n, err, fit[:, 0], fit[:, 1]


slopes = {}
plt.figure(figsize=(7, 4))
for (name, rows), color in zip(
    [("DTD", dtd_data), ("EuroSAT", eurosat_data)], ["tab:blue", "tab:orange"]
):
    n, err, beta, c = own_slopes(rows)
    slopes[name] = beta
    print(
        f"{name}: {int(n[0])} to {int(n[-1])} rows ({np.log10(n[-1] / n[0]):.1f} decades)"
    )
    print(f"  slope per seed {', '.join(f'{b:.2f}' for b in beta)}, {mean_std(beta)}")
    print(f"  10x the rows multiplies the error by {10 ** beta.mean():.2f}")
    plt.errorbar(
        n,
        err.mean(1),
        yerr=err.std(1),
        fmt="o",
        capsize=3,
        color=color,
        label=f"{name}, mean ± std over the seeds",
    )
    fit = 10 ** (c.mean() + beta.mean() * np.log10(n))
    plt.plot(
        n, fit, color=color, alpha=0.6, label=f"{name}, fit: slope {beta.mean():.2f}"
    )
d = slopes["EuroSAT"].mean() - slopes["DTD"].mean()
print(f"EuroSAT's mean slope minus DTD's: {d:+.3f}")
plt.xscale("log")
plt.yscale("log")
plt.xticks([150, 300, 1000, 3000], ["150", "300", "1000", "3000"])
plt.yticks(
    [0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.4],
    ["0.05", "0.07", "0.10", "0.15", "0.20", "0.30", "0.40"],
)
plt.minorticks_off()
plt.xlabel("training rows (log)")
plt.ylabel("test error, 1 - accuracy (log)")
plt.title("On log-log axes the two data axes fall with about the same slope")
plt.grid(alpha=0.3, which="both")
plt.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

**Reading.** DTD's slope is −0.23 ± 0.02 over the seeds, EuroSAT's −0.27 ± 0.02, and the
slopes of single seeds overlap (−0.21 to −0.25 on DTD, −0.24 to −0.30 on EuroSAT). On log
error the two data axes fall at about the same rate: 10× the rows multiplies DTD's error by
0.59 and EuroSAT's by 0.54.

DTD's accuracy moved 3.5 times as many points because its error is about four times larger:
0.20 against 0.05 at the full split. The same factor takes more points off a larger error. The
difference in points is explained by headroom; the slopes differ by no more than the spread
between their seeds.

These two datasets do not test the slope per domain of 3.2.6: both are image classification on
one frozen encoder.

**A local description, not a law.** The slopes cover 1.3 decades of rows, on a frozen encoder
with a small head: the encoder learned from far more images, in a pretraining set curated
towards DTD (E3), and only the head sees these rows. What 10× more rows brings is a
measurement too.

## What you now know

- A **scaling curve** is the score against one resource with the others held fixed, every
  point over a few seeds. The ± is the std over the seeds; a difference smaller than it could
  turn around with another seed.
- Choices (an epoch count, the head to prefer) are made on **valid**; the curves and every
  number quoted are scored on **test**, which no choice saw.
- On DTD with a frozen `dinov2-small`: width buys about 2.5 points and has flattened by
  `1 x 512`; compute is flat after epoch 20; data still pays 3.4 points at its last
  doubling.
- A conclusion can go beyond the top score: on valid, `1 x 256` lies close to the best head
  with a quarter of its parameters; read on test once, it costs about half a point and answers
  three to four times as fast on this machine, a price you state with the choice.
- A **scaling law** says the test loss falls as a power of compute, data or parameters while
  the other two are not the bottleneck: a straight line on log-log axes, whose slope says what
  10× more buys, inside the measured range only. It has been measured in language, images,
  speech and video, with a slope that differs per domain.
- On fixed data a bigger model's loss is still lower, but a larger share of it is lost to the
  limited data. Two questions get two ratios: how much data keeps that share equal (Kaplan: 8×
  the weights asks 4.6× the data), and how to split a fixed compute budget between weights and
  data (Kaplan 2.2×, Hoffmann about 8×, with every run trained to its end). A ratio belongs to
  the study that measured it.
- Your own data slopes were −0.23 ± 0.02 on DTD and −0.27 ± 0.02 on EuroSAT, no further apart
  than the spread between their seeds. DTD gained more accuracy points because its error was
  larger: the difference in points is headroom. Two image datasets on one encoder are one
  domain: a local description, not a law.

For the heartbeats, that means: measure the curve of a resource before you spend the budget on
it. The question to carry into notebook 03: if the axes that pay are not the ones you were
going to search over, **what should a search search?**

## Exercises

**E1: where does data stop paying?** Compare the last doubling of DTD and of EuroSAT, in
points of accuracy and on log error: does either curve bend at its end? Fit a line to accuracy
against log(rows) over the first three and over the last three points of each equal-steps
curve. At what gain per doubling would you stop labelling, and who decides that number?

**E2: does the parameter axis change with less data?** Repeat 3.2.2 on 10% of the train rows
(`rows=300` in `repeat`). Predict first: does capacity matter more when data is scarce, or
less? Then check the spread before believing the ranking.

**E3: the encoder axis.** Another encoder changes what the head reads, without a single
labelled row or epoch more. Load the DTD caches of `dinov2-small`, `resnet-18` and
`mobilenet_v2_1.0_224` (`load("dtd", encoder)`), train the baseline head on each with its own
epoch chosen on valid, and add `dinov2-base` and `dinov2-large`. How far apart are the
encoders, against the parameter axis of 3.2.2? Does the encoder with the largest head win?
Read the result with one fact: DINOv2's pretraining set holds a million web images retrieved
because they resemble DTD's training images (Oquab et al. 2023, Table 15). Near-duplicates of
test images were removed, so this is no leakage, but `dinov2-small` was curated towards
textures. Repeat on EuroSAT, which is not among its sources.

**E4: a curve that bends.** EuroSAT's last doubling gains 0.4 points, DTD's 3.4: does
EuroSAT's curve flatten below its straight line at the end? Fit
`error = gamma + alpha * rows**beta` instead (an offset `gamma` for the error no amount of
data removes, as Hestness et al. do) to the equal-steps curves of DTD and EuroSAT. Does the
offset explain a flattening? Extrapolate both fits to 10× the rows and argue, from the
residuals, which extrapolation you believe.

**E5: the budget of the data axis.** 3.2.3 gives every subset either `EPOCHS` epochs or the
steps of the full set. Give each subset its own epoch count instead, chosen on valid by the
rule of 3.2.1 (as 3.2.2 does per head), and draw that third curve next to the other two. Which
of the three would you report as "the data axis", and why?

**E6: read the ratios.** With the cells of 3.2.5: how much more data keeps the overfitting
penalty equal when the model grows 64×, by Kaplan's ratio? And how much goes with it in the
compute split of Kaplan, and of Hoffmann? Which would you use for the heads of 3.2.2, and why
may none of them apply?

**E7: latency beyond one machine.** 3.2.2 timed two heads over the whole test set as one
batch. Time them one row at a time as well. Does `1 x 1024` still take longer than `1 x 256`,
by more than the spread of the repeats? What would you have
to measure before claiming the same on a phone, and what before claiming anything about
energy?

**E8: depth on a frozen encoder.** Add heads of two to four hidden layers and train them as
in 3.2.2: 3008 rows, no regularisation, one learning rate for all. Do they beat one layer?
Then give them dropout or weight decay, and more rows (EuroSAT has 4800 train rows). What
does a deeper head need before its extra layers pay?